# Challenge de la Sesión 08 — Creación de clasificadores con modelos generativos

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/08-generative-classifiers/challenge.ipynb)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/08-generative-classifiers/challenge.ipynb)

> **Modalidad:** individual · **Tiempo:** 40 minutos en clase + entrega hasta las **23:59 de hoy** (hora de Colombia).
>
> **IA permitida con registro:** puedes usar asistentes de IA, pero debes completar el registro de uso de IA al final. Sin registro, el componente de reflexión y registro vale 0.0.
>
> **Micro-sustentaciones:** durante el bloque se eligen estudiantes al azar para explicar su notebook en 2–3 minutos. Si no puedes explicar tu entrega, la nota del challenge se limita a 3.0.
>
> **Rúbrica:** Ejecución técnica **40%** · Análisis e interpretación **40%** · Reflexión y registro de IA **20%**.
>
> **Entrega:** ejecuta todo en orden → `File → Download notebook` (Archivo → Descargar notebook) → súbelo a **e-Aulas** como **`S08_<codigo>.ipynb`**.

**Antes de empezar (Kaggle):** Settings → **Internet on**; Accelerator → **GPU T4 x2** recomendada (en CPU también corre, solo que el LLM es más lento).

**Contexto.** Como en el lab, clasificas tuits de noticias financieras (`zeroshot/twitter-financial-news-sentiment`) en **bajista** (*bearish*, 0), **alcista** (*bullish*, 1) o **neutral** (2). Tu configuración personal fija cuántos ejemplos etiquetados tienes, qué clase minoritaria vas a aumentar con datos sintéticos, tu porción de 60 tuits de prueba y la restricción de negocio de tu cliente (todas las empresas son ficticias).

In [ ]:
# ✏️ Escribe tu código estudiantil (como aparece en e-Aulas) y tu nombre completo.
STUDENT_ID = ""
STUDENT_NAME = ""

# --- Configuración del curso (no editar) ---
import os, sys, random, subprocess, importlib, hashlib, json
SESSION = "S08"
FAST_DEV_RUN = os.environ.get("COURSE_FAST_DEV_RUN") == "1"
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB = "google.colab" in sys.modules
if FAST_DEV_RUN and not STUDENT_ID.strip():
    STUDENT_ID, STUDENT_NAME = "000000", "Prueba automática"
assert STUDENT_ID.strip(), "⚠️ Escribe tu STUDENT_ID en la primera línea de esta celda y vuelve a ejecutarla."

def ensure(package, import_name=None):
    """Instala un paquete solo si falta."""
    try:
        importlib.import_module(import_name or package)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

PERSONAL_SEED = int(hashlib.sha256(f"{SESSION}:{STUDENT_ID.strip()}".encode()).hexdigest()[:8], 16)
rng = random.Random(PERSONAL_SEED)

def pick(options):
    """Elige una opción de forma determinística a partir de tu código estudiantil."""
    return rng.choice(list(options))

import socket
if os.environ.get("HF_HUB_OFFLINE") != "1":  # los modelos y datos se descargan de Hugging Face
    try:
        socket.create_connection(("huggingface.co", 443), timeout=10).close()
    except OSError:
        raise RuntimeError(
            "Sin conexión a internet: este notebook descarga modelos y datos de Hugging Face.\n"
            "Kaggle: en el panel derecho abre Settings (Session options) y activa Internet. "
            "La opción solo aparece si tu cuenta tiene el teléfono verificado (kaggle.com/settings). "
            "Luego vuelve a ejecutar esta celda. En Colab el internet viene activado."
        ) from None

import numpy as np
import torch
random.seed(PERSONAL_SEED); np.random.seed(PERSONAL_SEED % (2**32)); torch.manual_seed(PERSONAL_SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Sesión {SESSION} | Estudiante: {STUDENT_NAME} ({STUDENT_ID}) | Semilla personal: {PERSONAL_SEED} | Dispositivo: {DEVICE}")
# --- fin de la configuración ---
ensure("datasets")
ensure("transformers")
ensure("scikit-learn", "sklearn")

In [ ]:
# Tu configuración personal (no la cambies: depende de tu código estudiantil)
PERFILES = {
    "presupuesto": {
        "cliente": "Banco Andino, mesa de inversiones",
        "restriccion": "máximo USD 100 al mes para operar el clasificador (sin contar personal)",
        "volumen_diario": 20_000,
    },
    "latencia": {
        "cliente": "NovaTel Finanzas, alertas en su app de trading",
        "restriccion": "cada tuit debe clasificarse en menos de 50 ms, con picos de 50 tuits por segundo",
        "volumen_diario": 500_000,
    },
    "privacidad": {
        "cliente": "Seguros Cóndor, gestión de portafolio",
        "restriccion": "los textos incluyen notas internas y no pueden salir de su infraestructura (un servidor con una GPU pequeña)",
        "volumen_diario": 5_000,
    },
}
NOMBRES = {0: "bajista (bearish)", 1: "alcista (bullish)", 2: "neutral"}

_inicio = pick(range(0, 2340, 60))
CONFIG = {
    "n_train": pick([30, 60, 120, 240]),          # ejemplos etiquetados para entrenar
    "minority_class": pick([0, 1]),               # clase que aumentarás con datos sintéticos
    "test_slice": [_inicio, _inicio + 60],        # tus 60 tuits de prueba (del split de validación)
    "constraint_profile": pick(sorted(PERFILES)),
}
PERFIL = PERFILES[CONFIG["constraint_profile"]]
MINORITARIA = CONFIG["minority_class"]

print("🎯 Tu configuración personal")
print(f"  • Ejemplos de entrenamiento (n_train): {CONFIG['n_train']}")
print(f"  • Clase minoritaria a aumentar: {MINORITARIA} = {NOMBRES[MINORITARIA]}")
print(f"  • Porción de prueba: tuits {CONFIG['test_slice'][0]} a {CONFIG['test_slice'][1] - 1} del split de validación barajado")
print(f"  • Perfil de restricción: {CONFIG['constraint_profile']} → {PERFIL['cliente']}: {PERFIL['restriccion']} "
      f"(≈ {PERFIL['volumen_diario']:,} tuits al día)")

### Datos (ejecuta sin cambios)
Tu conjunto de entrenamiento es una muestra estratificada de `n_train` tuits del split `train`; tu conjunto de prueba son 60 tuits consecutivos del split `validation` barajado con una semilla fija. **Nunca** entrenes ni ajustes nada con tu porción de prueba.

In [ ]:
import time, re
import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

ETIQUETAS = ["bearish", "bullish", "neutral"]
N_LLM = 12 if FAST_DEV_RUN else 60                  # tuits de prueba que califica el LLM

raw = load_dataset("zeroshot/twitter-financial-news-sentiment")
train_full = raw["train"].shuffle(seed=0).to_pandas()
val_full = raw["validation"].shuffle(seed=0).to_pandas()

def muestra_estratificada(df, n, seed):
    """n filas con las proporciones de clase de df (mínimo 2 por clase)."""
    rs = np.random.RandomState(seed)
    props = df.label.value_counts(normalize=True).sort_index()
    idx = []
    for c, p in props.items():
        k = max(2, int(round(n * p)))
        idx += list(rs.choice(np.where(df.label.values == c)[0], k, replace=False))
    return df.iloc[idx].reset_index(drop=True)

train_df = muestra_estratificada(train_full, CONFIG["n_train"], seed=PERSONAL_SEED % (2**32))
i0, i1 = CONFIG["test_slice"]
test_df = val_full.iloc[i0:i1].reset_index(drop=True)
print("Entrenamiento:", dict(zip(ETIQUETAS, np.bincount(train_df.label, minlength=3))))
print("Prueba       :", dict(zip(ETIQUETAS, np.bincount(test_df.label, minlength=3))))

def metricas(y_true, y_pred, clase=MINORITARIA):
    """Accuracy, macro-F1 y F1 de la clase minoritaria asignada."""
    rep = classification_report(y_true, y_pred, labels=[0, 1, 2], output_dict=True, zero_division=0)
    return {"accuracy": round(accuracy_score(y_true, y_pred), 4),
            "macro_f1": round(f1_score(y_true, y_pred, labels=[0, 1, 2], average="macro", zero_division=0), 4),
            "f1_minoritaria": round(rep[str(clase)]["f1-score"], 4),
            "recall_minoritaria": round(rep[str(clase)]["recall"], 4)}

## Tarea 1 — Naive Bayes vs. regresión logística vs. LLM zero-shot en tu porción (Ejecución técnica)

Entrena **Naive Bayes** (generativo) y **regresión logística** (discriminativo) sobre TF-IDF con tus `n_train` ejemplos, y compáralos con el **LLM zero-shot** (`Qwen/Qwen2.5-0.5B-Instruct`) que clasifica puntuando la log-verosimilitud de cada etiqueta, como en el lab (regla: suma de log-probabilidades). Mide también la latencia por tuit.

Los hiperparámetros tienen valores razonables; puedes cambiarlos si lo justificas en tu respuesta, pero **nunca** eligiéndolos según el resultado en tu porción de prueba.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ (opcional): ajusta los hiperparámetros y justifícalo en tu respuesta
ALPHA_NB = 0.1       # suavizado de Laplace de Naive Bayes
C_LR = 10.0          # inverso de la regularización L2 de la regresión logística

vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
X_train = vec.fit_transform(train_df.text)           # el vectorizador se ajusta SOLO con entrenamiento
X_test = vec.transform(test_df.text)
nb = MultinomialNB(alpha=ALPHA_NB).fit(X_train, train_df.label)
lr = LogisticRegression(C=C_LR, max_iter=2000).fit(X_train, train_df.label)

def ms_por_tuit(fn, n):
    t0 = time.perf_counter(); salida = fn(); return salida, 1000 * (time.perf_counter() - t0) / n

pred_nb, ms_nb = ms_por_tuit(lambda: nb.predict(vec.transform(test_df.text)), len(test_df))
pred_lr, ms_lr = ms_por_tuit(lambda: lr.predict(vec.transform(test_df.text)), len(test_df))
t1 = {"Naive Bayes": {**metricas(test_df.label, pred_nb), "ms_por_tuit": round(ms_nb, 3)},
      "Regresión logística": {**metricas(test_df.label, pred_lr), "ms_por_tuit": round(ms_lr, 3)}}
pd.DataFrame(t1).T

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME).float().to(DEVICE).eval()   # float32: rápido en CPU y T4
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
VERBALIZADORES = ["bearish", "bullish", "neutral"]
IDS_ETIQUETAS = [tokenizer(v, add_special_tokens=False).input_ids for v in VERBALIZADORES]

def mensajes(tuit):
    return [{"role": "system", "content": "You are a financial analyst. You label the sentiment of financial news tweets."},
            {"role": "user", "content": f"Tweet: {tuit}\n\nIs this tweet bearish, bullish or neutral for the "
                                        "stock or market it mentions? Answer with one lowercase word: "
                                        "bearish, bullish or neutral."}]

@torch.no_grad()
def puntajes_etiquetas(tuit):
    """Suma de log-probabilidades de los tokens de cada etiqueta después del prompt."""
    p = tokenizer.apply_chat_template(mensajes(tuit), add_generation_prompt=True,
                                      return_tensors="pt", return_dict=True)["input_ids"][0]
    seqs = [torch.cat([p, torch.tensor(ids)]) for ids in IDS_ETIQUETAS]
    L = max(len(s) for s in seqs)
    input_ids = torch.full((len(seqs), L), PAD_ID)
    atencion = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        input_ids[i, :len(s)] = s
        atencion[i, :len(s)] = 1
    logits = llm(input_ids=input_ids.to(DEVICE), attention_mask=atencion.to(DEVICE)).logits
    puntajes = []
    for i, ids in enumerate(IDS_ETIQUETAS):
        pos = torch.arange(len(p) - 1, len(p) - 1 + len(ids), device=logits.device)   # desplazamiento de uno
        objetivo = torch.tensor(ids, device=logits.device)
        puntajes.append(logits[i, pos].float().log_softmax(-1).gather(1, objetivo[:, None]).sum().item())
    return puntajes

llm_df = test_df.iloc[:N_LLM]
t0 = time.perf_counter()
puntajes_llm = np.array([puntajes_etiquetas(t) for t in llm_df.text])
ms_llm = 1000 * (time.perf_counter() - t0) / len(llm_df)
pred_llm = puntajes_llm.argmax(1)
t1["LLM zero-shot"] = {**metricas(llm_df.label, pred_llm), "ms_por_tuit": round(ms_llm, 3)}
if len(llm_df) < len(test_df):
    print(f"(Modo rápido: el LLM solo calificó {len(llm_df)} tuits.)")
print("Etiquetas predichas por el LLM:", dict(zip(ETIQUETAS, np.bincount(pred_llm, minlength=3))))
tabla_t1 = pd.DataFrame(t1).T
tabla_t1

**Tu respuesta:**
1. ¿Qué modelo gana en tu porción con tu `n_train`? Cita accuracy, macro-F1 y el F1 de tu clase minoritaria de la tabla.
2. ¿Tu resultado coincide con lo que predice Ng y Jordan (2002) para tu tamaño de entrenamiento? Relaciónalo con la curva de aprendizaje del lab.
3. ¿Cuántos tuits de tu clase minoritaria hay en tu porción de 60? ¿Qué tan confiable es un F1 calculado con tan pocos casos?

*(Escribe aquí.)*

## Tarea 2 — Aumento sintético para tu clase minoritaria (Análisis e interpretación)

Genera ejemplos **sintéticos** de tu clase minoritaria con el LLM (semillas: tuits reales de tu entrenamiento), límpialos y deduplícalos, **léelos**, agrégalos **solo al entrenamiento** y compara el antes y el después en tu porción de prueba real.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ (opcional): puedes cambiar la cantidad, la temperatura o el prompt
N_SINTETICOS = 4 if FAST_DEV_RUN else 20
TEMPERATURA = 0.9
MAX_TOKENS = 24 if FAST_DEV_RUN else 48

def limpiar(texto):
    lineas = [ln.strip() for ln in texto.strip().splitlines() if ln.strip()]
    if not lineas:
        return ""
    t = re.sub(r"^(tweet\s*:|[-*•]\s*|\d+[.)]\s*)", "", lineas[0], flags=re.I).strip()
    return t.strip('"“”\' ')

def generar(nombre_clase, semillas, n, temperatura=TEMPERATURA, por_llamada=10):
    rs = np.random.RandomState(PERSONAL_SEED % (2**32))
    torch.manual_seed(PERSONAL_SEED)
    salidas = []
    while len(salidas) < n:
        ejemplos = "\n".join(f"- {s}" for s in rs.choice(semillas, size=min(3, len(semillas)), replace=False))
        msgs = [{"role": "system", "content": "You write realistic, short financial news tweets."},
                {"role": "user", "content": f"Here are examples of {nombre_clase} financial news tweets:\n{ejemplos}\n\n"
                                            f"Write one new {nombre_clase} financial news tweet about a different "
                                            "company. Output only the tweet."}]
        enc = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(DEVICE)
        k = min(por_llamada, n - len(salidas))
        out = llm.generate(**enc, do_sample=True, temperature=temperatura, top_p=0.95, max_new_tokens=MAX_TOKENS,
                           num_return_sequences=k, pad_token_id=tokenizer.eos_token_id)
        salidas += tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return salidas

semillas = train_df[train_df.label == MINORITARIA].text.values
t0 = time.perf_counter()
generados = generar(VERBALIZADORES[MINORITARIA], semillas, N_SINTETICOS)
vistos = {t.lower() for t in train_full.text}
sinteticos = []
for g in generados:
    t = limpiar(g)
    if len(t.split()) >= 4 and t.lower() not in vistos:
        sinteticos.append(t); vistos.add(t.lower())
print(f"Generados {len(generados)} en {time.perf_counter() - t0:.1f} s; conservados {len(sinteticos)} tras limpiar y deduplicar:\n")
for i, t in enumerate(sinteticos, 1):
    print(f"{i:>2}. {t}")

In [ ]:
def entrenar_y_evaluar(textos, etiquetas):
    v = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
    Xtr = v.fit_transform(textos)
    Xte = v.transform(test_df.text)
    out = {}
    for nombre, modelo in [("NB", MultinomialNB(alpha=ALPHA_NB)), ("LR", LogisticRegression(C=C_LR, max_iter=2000))]:
        out[nombre] = metricas(test_df.label, modelo.fit(Xtr, etiquetas).predict(Xte))
    return out

antes = entrenar_y_evaluar(list(train_df.text), list(train_df.label))
despues = entrenar_y_evaluar(list(train_df.text) + sinteticos, list(train_df.label) + [MINORITARIA] * len(sinteticos))
tabla_t2 = pd.DataFrame({f"{m} · {cuando}": d[m] for cuando, d in [("solo reales", antes), ("reales + sintéticos", despues)]
                         for m in ["NB", "LR"]}).T
tabla_t2

**Tu respuesta:**
1. ¿Mejoró el F1 (y el recall) de tu clase minoritaria con regresión logística? ¿Y el macro-F1? Cita los valores antes y después.
2. Lee tus ejemplos sintéticos: ¿cuántos están mal etiquetados (no son de tu clase) o son repetitivos? Da dos ejemplos concretos.
3. Con solo 60 tuits de prueba, ¿la diferencia es convincente? ¿Qué harías para estar más seguro(a) antes de usar datos sintéticos en producción?

*(Escribe aquí.)*

## Tarea 3 — Explica y decide: tu recomendación bajo restricciones (Análisis e interpretación)

Tu cliente y su restricción están en tu configuración (`PERFIL`). La celda siguiente estima el costo mensual de un LLM por API frente a un modelo propio. **Reemplaza los precios de ejemplo** por precios actuales de un proveedor real y anota la fuente y la fecha en tu respuesta.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ: reemplaza los precios de ejemplo (son marcadores, no cotizaciones reales)
PRECIO_API_USD_POR_MILLON_TOKENS = 1.00
COSTO_SERVIDOR_USD_MES = 400.0

def n_tokens_prompt(tuit):
    return tokenizer.apply_chat_template(mensajes(tuit), add_generation_prompt=True,
                                         return_tensors="pt", return_dict=True)["input_ids"].shape[1]

tokens_por_tuit = np.mean([n_tokens_prompt(t) for t in llm_df.text]) + 2      # prompt + respuesta corta
volumen = PERFIL["volumen_diario"]
costo_api_mes = volumen * 30 * tokens_por_tuit * PRECIO_API_USD_POR_MILLON_TOKENS / 1e6
print(f"Cliente: {PERFIL['cliente']} | Restricción: {PERFIL['restriccion']}")
print(f"Volumen: {volumen:,} tuits/día · ≈ {tokens_por_tuit:.0f} tokens por tuit con el prompt del LLM")
print(f"LLM por API (estimado): USD {costo_api_mes:,.0f} al mes · Modelo propio (un servidor): USD {COSTO_SERVIDOR_USD_MES:,.0f} al mes")
print("Latencia medida en este entorno (ms por tuit):",
      {m: t1[m]["ms_por_tuit"] for m in t1}, f"| dispositivo: {DEVICE}")

**Tu respuesta:** recomienda **un** clasificador para tu cliente. Estructura sugerida (8–12 líneas):
1. **Decisión:** qué modelo (NB, regresión logística, MLP sobre embeddings, LLM local, LLM por API…) y dónde corre.
2. **Evidencia:** al menos **tres de tus números** (calidad, latencia y costo) de las tareas 1–3.
3. **Restricción:** cómo cumple la restricción de tu perfil y qué alternativa descartas y por qué.
4. **Riesgo principal** (por ejemplo, calibración, sesgo del LLM, datos sintéticos, pocos datos de prueba) y cómo lo mitigarías.
5. **Siguiente paso** en las próximas dos semanas (por ejemplo, cuántos tuits etiquetarías y para qué).

*(Escribe aquí.)*

## Tarea 4 — Crítica a la IA: fugas de datos (Análisis e interpretación)

Le pedimos a un asistente de IA: *"Escribe código para evaluar una alerta que detecte mi clase minoritaria con regresión logística, usando también mis ejemplos sintéticos"*. Este fue su código. Reporta un F1 que parece prometedor, pero contiene **tres errores de fuga de datos** (*data leakage*). Ejecútalo y luego critícalo.

In [ ]:
# 🤖 Código generado por un asistente de IA (contiene errores: NO lo tomes como modelo)
from sklearn.model_selection import train_test_split

textos = list(train_df.text) + list(test_df.text) + sinteticos
etiquetas = list(train_df.label) + list(test_df.label) + [MINORITARIA] * len(sinteticos)
y_alerta = np.array([int(e == MINORITARIA) for e in etiquetas])          # 1 = alerta (clase minoritaria)

vec_ia = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
X_todo = vec_ia.fit_transform(textos)                                     # "así el vocabulario queda completo"

X_tr_ia, X_te_ia, y_tr_ia, y_te_ia = train_test_split(X_todo, y_alerta, test_size=0.3, random_state=0)
clf_ia = LogisticRegression(C=10, max_iter=2000).fit(X_tr_ia, y_tr_ia)
p_ia = clf_ia.predict_proba(X_te_ia)[:, 1]

umbrales = np.arange(0.05, 0.95, 0.05)
f1s = [f1_score(y_te_ia, p_ia >= u, zero_division=0) for u in umbrales]
umbral_ia = float(umbrales[int(np.argmax(f1s))])
f1_ia = float(max(f1s))
print(f"F1 de la alerta: {f1_ia:.3f} con umbral {umbral_ia:.2f}  ← 'listo para producción', según el asistente")

**Tu respuesta:** para cada uno de los **tres errores**: (a) cita la línea, (b) explica por qué es una fuga y en qué dirección sesga el F1, y (c) di cómo se corrige.

*(Escribe aquí.)*

Ahora **corrige** el código en la celda siguiente. El esqueleto corre tal cual, pero **todavía tiene las tres fugas** en las líneas marcadas con ✏️. Reglas: el vectorizador y el modelo se ajustan solo con entrenamiento (reales + sintéticos); la evaluación se hace **una sola vez** sobre tu porción real `test_df`; el umbral se elige **sin mirar** el test (pista: `cross_val_predict` sobre el entrenamiento).

In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold

def a_alerta(etiquetas_multiclase):
    return np.array([int(e == MINORITARIA) for e in etiquetas_multiclase])

# ✏️ TU CÓDIGO AQUÍ: corrige las tres líneas marcadas con (1), (2) y (3)
textos_ent = list(train_df.text) + sinteticos + list(test_df.text)                                       # ✏️ (2)
y_ent = a_alerta(list(train_df.label) + [MINORITARIA] * len(sinteticos) + list(test_df.label))           # ✏️ (2)
textos_test, y_test = list(test_df.text), a_alerta(test_df.label)

vec_ok = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
vec_ok.fit(textos_ent + textos_test)                                                                     # ✏️ (1)
X_ent, X_test_ok = vec_ok.transform(textos_ent), vec_ok.transform(textos_test)
clf_ok = LogisticRegression(C=10, max_iter=2000).fit(X_ent, y_ent)
p_test = clf_ok.predict_proba(X_test_ok)[:, 1]

umbrales = np.arange(0.05, 0.95, 0.05)
p_para_umbral, y_para_umbral = p_test, y_test                                                            # ✏️ (3)
umbral_ok = float(umbrales[int(np.argmax([f1_score(y_para_umbral, p_para_umbral >= u, zero_division=0) for u in umbrales]))])
f1_corregido = float(f1_score(y_test, p_test >= umbral_ok, zero_division=0))
print(f"F1 corregido de la alerta en tu porción real: {f1_corregido:.3f} con umbral {umbral_ok:.2f} (IA: {f1_ia:.3f})")

**Tu respuesta:** ¿cuánto cambió el F1 al corregir las fugas? ¿Cuál de los tres errores crees que infló más el resultado y por qué? ¿Qué le dirías a un gerente que ya vio el F1 del asistente?

*(Escribe aquí.)*

In [ ]:
RESULTS = {
    # la latencia (ms_por_tuit) varía entre ejecuciones: se deja fuera de la huella
    "t1_nb": {k: v for k, v in t1["Naive Bayes"].items() if k != "ms_por_tuit"},
    "t1_lr": {k: v for k, v in t1["Regresión logística"].items() if k != "ms_por_tuit"},
    "t1_llm": {k: v for k, v in t1["LLM zero-shot"].items() if k != "ms_por_tuit"},
    "t2_sinteticos_conservados": len(sinteticos),
    "t2_lr_antes": antes["LR"],
    "t2_lr_despues": despues["LR"],
    "t4_f1_ia": round(f1_ia, 4),
    "t4_f1_corregido": round(f1_corregido, 4),
    "t4_umbral_corregido": round(umbral_ok, 4),
}
print(json.dumps(RESULTS, indent=2, ensure_ascii=False))

## 🪞 Reflexión y registro de uso de IA (20%)

### Reflexión (responde en 3–5 líneas cada una)
1. ¿Qué aprendiste hoy que puedas aplicar en tu trabajo u organización?
2. ¿Qué fue lo más difícil y cómo lo resolviste?
3. Autoevaluación: ¿qué nota (0.0–5.0) te pondrías en este challenge y por qué?

### 🤖 Registro de uso de IA (obligatorio)
| # | Herramienta (ChatGPT, Claude, Codex, Copilot…) | ¿Para qué la usaste? | Prompt principal (resumido) | ¿Qué verificaste o corregiste de su respuesta? |
|---|---|---|---|---|
| 1 |  |  |  |  |
| 2 |  |  |  |  |

> Si no usaste asistentes de IA, escribe: *"No usé asistentes de IA en este challenge."* Recuerda: puedes usar IA, pero **eres responsable de todo lo que entregas** y podrías ser seleccionado(a) para una micro-sustentación.

In [ ]:
# --- Huella de resultados (no editar) ---
def results_fingerprint(results):
    payload = json.dumps({"session": SESSION, "student": STUDENT_ID.strip(), "results": results},
                         sort_keys=True, default=str, ensure_ascii=False)
    return hashlib.sha256(payload.encode()).hexdigest()[:12]

print(json.dumps({"CONFIG": CONFIG, "RESULTS": RESULTS}, indent=2, ensure_ascii=False, default=str))
print("🔏 Huella de resultados:", results_fingerprint({"CONFIG": CONFIG, "RESULTS": RESULTS}))